In [1]:
import sys
import warnings 
from pathlib import Path
from collections import defaultdict
sys.path.append(str(Path.cwd().parents[1]))

import pandas as pd
import jax.numpy as jnp

from source.exp_functions import CVTracker
from configs.uci import FMAP, DATASETS, CV_NAME, RES_NAME, PRED_NAME

warnings.filterwarnings("ignore")

ART_DIR = Path('./artifacts/training_artifacts')

In [2]:
def get_res_df(
    model: str, 
    hess: str, 
    data_list: list[str], 
    fmap: str = FMAP,
    metric: str = 'nll_test',
    mean_dec: int = 2,
    std_dec: int = 2,
) -> pd.DataFrame:
    """ Get pandas dataframe with metric results """
    dd = defaultdict(list)
    for data in data_list:
        dd['data'].append(data if data != "wine_red" else "red wine")
        res_dir = ART_DIR / f'{model}/{data}/{hess}/ll/{fmap}/'
        try:
            tracker = CVTracker(res_dir, RES_NAME, CV_NAME, PRED_NAME)
            res_df, _, _ = tracker.load()
            _mean, _std = res_df[metric].mean(), res_df[metric].std()
            if _mean is jnp.nan or (metric == 'ecp_test' and _mean == 0):
                dd[model.upper()].append('N/A±0.00')
            else:
                dd[model.upper()].append(
                    mean_std_str(_mean, _std, mean_dec=mean_dec, std_dec=std_dec)
                )
        except:
            dd[model.upper()].append('N/A±0.00')
    return pd.DataFrame(dd).rename({'LA_BTN': 'LA-TNKM'}, axis=1)

def mean_std_str(mean, std, mean_dec=2, std_dec=2):
    return f"{mean:.{mean_dec}f}±{std:.{std_dec}f}"

def get_metric_results(
    metric: str, 
    use_gwi_table: bool = True,
    mean_dec: int = 2,
    std_dec: int = 2,
) -> pd.DataFrame:
    uci_gwi = pd.read_csv('./extra_data/uci_gwi_paper_table.csv', index_col=0)
    uci_gwi = uci_gwi if use_gwi_table else uci_gwi['Dataset']
    models = ['la_bnn', 'mf_btn', 'sp_btn', 'la_btn']
    hess_list = ['full', 'mf', 'mf', 'last']
    for model, hess in zip(models, hess_list):
        fmap = 'alt' if model == 'la_bnn' else FMAP
        res_df = get_res_df(model, hess, DATASETS, fmap, metric, mean_dec, std_dec)
        uci_gwi = pd.merge(uci_gwi, res_df, left_on='Dataset', right_on='data')
        uci_gwi = uci_gwi.drop('data', axis=1)
    uci_gwi['Dataset'] = uci_gwi['Dataset'].apply(lambda x: x.upper())
    return uci_gwi

def bold_row_min(row):
    def _str(s):
        return isinstance(s, str) and 'N/A' not in s and '±' in s
    # Extract mean values from strings
    means = [float(s.split('±')[0]) if _str(s) else 1e10 for s in row]
    min_mean = min(means)
    # Bold the minimum(s)
    return pd.Series(
        [f"\\textbf{{{s}}}" if _str(s) and (float(s.split('±')[0]) == min_mean) else s for s in row],
        index=row.index,
    )

### NLL (Test):

In [4]:
metric = 'nll_test'
res_df = get_metric_results(metric, use_gwi_table=True)
res_df = res_df.rename({'GWI DNN-SVGP': 'GWI-DNN'}, axis=1)
res_df = res_df.drop(['FBNN', 'VIP-BNN'], axis=1)

In [ ]:
TEXT_BF_BOOL = True

if TEXT_BF_BOOL:
    res_df = (
        res_df
        .set_index("Dataset")
        .apply(bold_row_min, axis=1)
        .reset_index()
    )

caption = (
    r"Average test NLL on several UCI regression datasets. "
    + r"We train on random 90\% of the data and predict on 10\%. "
    + r"Each experiment is repeated 10 times, and we report the mean ± standard deviation. "
    + r"$N$ is the sample size, and $D$ is the data dimensionality. "
    + r"'N/A' refers to cases where reporting a value is not feasible due to "
    + r"prohibitively large computational demands or lack of numerical stability. "
    + r"Among the evaluated methods, LA-TNKM (Last) ranks highest on four of the nine "
    + r"datasets and performs comparably on the rest."
)
print(
    (
        res_df
        .to_latex(
            float_format="%.2f", 
            column_format='||l|c|c||c|c|c|c|c|c|c|c||c||',
            caption=caption,
            label="table:uci-comparison",
            multirow=False,
            index=False,
        ).replace('_', '-')
    )
);

### RMSE (Test) and RCE (Test):

In [6]:
TEXT_BF_BOOL = True
USE_DOWNARROW = True

darr = '$\downarrow$' if USE_DOWNARROW else ''
metric = 'rmse_test'
res_df_ecp = get_metric_results(metric, use_gwi_table=False)
metric = 'rce_test'
res_df_wcpi = get_metric_results(metric, use_gwi_table=False)

df_a = res_df_ecp.set_index("Dataset")
df_b = res_df_wcpi.set_index("Dataset")

if TEXT_BF_BOOL:
    df_a = df_a.apply(bold_row_min, axis=1)
    df_b = df_b.apply(bold_row_min, axis=1)

df_a.columns = pd.MultiIndex.from_product(
    [[f'RMSE {darr}'], df_a.columns]
)
df_b.columns = pd.MultiIndex.from_product(
    [[f'RCE {darr}'], df_b.columns]
)
df_merged = pd.concat([df_a, df_b], axis=1)

In [ ]:
caption = (
    r"Average test RMSE and RCE on several UCI regression datasets. "
    + r"We train on random 90\% of the data and predict on 10\%. "
    + r"Each experiment is repeated 10 times, and we report the mean ± standard deviation. "
)
print(
    (
        df_merged
        .to_latex(
            float_format="%.2f", 
            column_format='||l||c|c|c|c||c|c|c|c||',
            caption=caption,
            label="table:uci_rmse_rce",
            multicolumn=True, 
            multicolumn_format='c||',
            multirow=True,
            index=True,
        ).replace('_', '-')
    )
);

### ECP-95 (Test) and WCPI-95 (Test):

In [8]:
TEXT_BF_BOOL = True
USE_DOWNARROW = True

darr = '$\downarrow$' if USE_DOWNARROW else ''
metric = 'ecp_test'
res_df_ecp = get_metric_results(metric, use_gwi_table=False)
metric = 'wcpi_test'
res_df_wcpi = get_metric_results(metric, use_gwi_table=False)

df_a = res_df_ecp.set_index("Dataset")
df_b = res_df_wcpi.set_index("Dataset")
if TEXT_BF_BOOL:
    df_b = df_b.apply(bold_row_min, axis=1)
    
df_a.columns = pd.MultiIndex.from_product(
    [["ECP-95"], df_a.columns]
)
df_b.columns = pd.MultiIndex.from_product(
    [[f"WCPI-95 {darr}"], df_b.columns]
)
df_merged = pd.concat([df_a, df_b], axis=1)

In [ ]:
caption = (
    r"Average test ECP-95 and WCPI-95 on several UCI regression datasets. "
    + r"We train on random 90\% of the data and predict on 10\%. "
    + r"Each experiment is repeated 10 times, and we report the mean ± standard deviation. "
    + r"'N/A' refers to cases where reporting a value is not feasible due to "
    + r"prohibitively large computational demands or lack of numerical stability. "
)
print(
    (
        df_merged
        .to_latex(
            float_format="%.2f", 
            column_format='||l||c|c|c|c||c|c|c|c||',
            caption=caption,
            label="table:uci_ecp_wcpi",
            multicolumn=True, 
            multicolumn_format='c||',
            multirow=True,
            index=True,
        ).replace('_', '-')
    )
);

### LA-TNKM - different Hessian (covariance) matrix approximations:

In [10]:
TEXT_BF_BOOL = True
USE_DOWNARROW = True
mean_dec, std_dec = 2, 2

darr = '$\downarrow$' if USE_DOWNARROW else ''
models = ['la_btn', 'la_btn', 'la_btn']
hess_list = ['gauss_newton', 'block', 'last']
hess2name = dict(gauss_newton='GNN', block='Block', last='Last')
metrics = ['rmse_test', 'nll_test', 'rce_test']
metric2name = dict(rmse_test=f'RMSE {darr}', nll_test=f'NLL {darr}', rce_test=f'RCE {darr}')

res = []
for metric in metrics:
    uci_gwi = pd.read_csv(
        './extra_data/uci_gwi_paper_table.csv', index_col=0
    )['Dataset']
    for model, hess in zip(models, hess_list):
        fmap = 'alt' if model == 'la_bnn' else FMAP
        res_df = get_res_df(model, hess, DATASETS, fmap, metric, mean_dec, std_dec)
        res_df = res_df.rename({'LA-TNKM': hess2name[hess]}, axis=1)
        uci_gwi = pd.merge(uci_gwi, res_df, left_on='Dataset', right_on='data')
        uci_gwi = uci_gwi.drop('data', axis=1)
    uci_gwi['Dataset'] = uci_gwi['Dataset'].apply(lambda x: x.upper())
    uci_gwi = uci_gwi.set_index("Dataset")
    if TEXT_BF_BOOL:
        uci_gwi = uci_gwi.apply(bold_row_min, axis=1)
    uci_gwi.columns = pd.MultiIndex.from_product(
        [[metric2name[metric]], uci_gwi.columns]
    )
    res.append(uci_gwi)

df_merged = pd.concat(res, axis=1)

In [ ]:
caption = (
    r"Average test RMSE, NLL, and RCE on several UCI regression datasets. "
    + r"Models are trained on 90\% of the data and tested on the remaining 10\%. "
    + r"Each experiment is repeated 10 times, and we report mean ± standard deviation "
    + r"for three Hessian matrix approximations for LA-TNKM: GNN, Block, and Last. "
)
print(
    (
        df_merged
        .to_latex(
            float_format="%.2f", 
            column_format='||l||c|c|c||c|c|c||c|c|c||',
            caption=caption,
            label="table:uci_hessian",
            multicolumn=True, 
            multicolumn_format='c||',
            multirow=True,
            index=True,
        ).replace('_', '-')
    )
);